In [1]:
import json
import pickle 
from src.gpt_model import Model
import ast


DIMENSION = "coherence"
coherence_rubric = 'Coherence - the collective quality of all sentences. We align this dimension with the DUC quality question of structure and coherence whereby "the summary should be well-structured and well-organized. The summary should not just be a heap of related information, but should build from sentence to a coherent body of information about a topic."'
consistency_rubric = "Consistency - the factual alignment between the summary and the summarized source. A factually consistent summary contains only statements that are entailed by the source document. Annotators were also asked to penalize summaries that contained hallucinated facts. "
fluency_rubric = 'Fluency - the quality of individual sentences. Drawing again from the DUC quality guidelines, sentences in the summary "should have no formatting problems, capitalization errors or obviously ungrammatical sentences (e.g., fragments, missing components) that make the text difficult to read."'
relevance_rubric = "Relevance - selection of important content from the source. The summary should include only important information from the source document. Annotators were instructed to penalize summaries which contained redundancies and excess information."

DIMENSION_RUBRIC = {
    "coherence": coherence_rubric,
    "consistency": consistency_rubric, 
    "fluency": fluency_rubric,
    "relevance": relevance_rubric
}

with open(f"./data/summeval/{DIMENSION}/human_llm_attributes.txt", "r") as file:
    attributes_list = file.read().splitlines()
attributes = "\n".join(attributes_list)

# LLM Initiation

In [2]:
with open("./api_keys.json", "r") as file:
    api_keys = json.load(file)

OPENAI_API_KEY = api_keys["openai"]

gpt4 = Model(model="gpt-4", temperature=0.0, api_key=OPENAI_API_KEY)

gpt-4


# Component Extraction

# Prompt Construction

In [4]:
with open(f"./prompts/summeval/checklist_construction/component_extraction/system_prompt.txt", "r") as file:
    sys_prompt = file.read()

with open("./prompts/summeval/checklist_construction/component_extraction/user_prompt.txt", "r") as file:
    user_prompt = file.read()

# Component Generation

In [5]:
prompt_list = [
    {"role":"system", "content": sys_prompt.format(DIMENSION, DIMENSION, DIMENSION_RUBRIC[DIMENSION])},
    {"role": "user", "content": user_prompt.format(attributes)}
]

gpt4_response = gpt4.ask_chatgpt(prompt_list)
components = ast.literal_eval(gpt4_response)

In [6]:
components

['Well-structured and well-organized summary',
 'Logical flow of ideas and events',
 'Clear and concise representation of main points',
 'Consistent context and meaning',
 'Complete overview of the original content']

# Attributes Clustering

In [7]:
with open("./prompts/summeval/checklist_construction/attributes_clustering/system_prompt.txt", "r") as file:
    sys_prompt = file.read()

with open("./prompts/summeval/checklist_construction/attributes_clustering/user_prompt.txt", "r") as file:
    user_prompt = file.read()

In [8]:
prompt_list = [
    {"role":"system", "content": sys_prompt},
    {"role": "user", "content": user_prompt.format(components, attributes)}
]

gpt4_response = gpt4.ask_chatgpt(prompt_list)
components_attributes_dic = eval(gpt4_response)

In [9]:
components_attributes = ""
for k, v in components_attributes_dic.items():
    components_attributes += f"{k}:\n{v}\n\n"
    

# Key Question Generation

In [10]:
with open("./prompts/summeval/checklist_construction/question_generation/system_prompt.txt", "r") as file:
    sys_prompt = file.read()
with open("./prompts/summeval/checklist_construction/question_generation/user_prompt.txt", "r") as file:
    user_prompt = file.read()

In [11]:
prompt_list = [
    {"role":"system", "content": sys_prompt.format(DIMENSION)},
    {"role": "user", "content": user_prompt.format(DIMENSION, DIMENSION, DIMENSION_RUBRIC[DIMENSION], components_attributes)}
]

generated_key_questions = gpt4.ask_chatgpt(prompt_list)

generated_key_questions = eval(generated_key_questions)

In [12]:
key_questions = ""
for component, question in generated_key_questions.items():
    key_questions += "- "+component+": "+question+"\n"

# Sub-question Generation

In [13]:
with open("./prompts/summeval/checklist_construction/sub_question_generation/system_prompt.txt", "r") as file:
    sys_prompt = file.read()
with open("./prompts/summeval/checklist_construction/sub_question_generation/user_prompt.txt", "r") as file:    
    user_prompt = file.read()

In [14]:
prompt_list = [
    {"role":"system", "content": sys_prompt},
    {"role": "user", "content": user_prompt.format(DIMENSION, DIMENSION, DIMENSION, DIMENSION_RUBRIC[DIMENSION], key_questions)}
]
generated_sub_questions = gpt4.ask_chatgpt(prompt_list)
generated_sub_questions = eval(generated_sub_questions)

In [15]:
sub_questions = ""
for component, sub_question_list in generated_sub_questions.items():
    for sub_question in sub_question_list:
        sub_questions += f"- {sub_question}\n"

# Question Validation

In [16]:
with open("./prompts/summeval/checklist_construction/question_validation/system_prompt.txt", "r") as file:
    sys_prompt = file.read()
with open("./prompts/summeval/checklist_construction/question_validation/user_prompt.txt", "r") as file:    
    user_prompt = file.read()

In [17]:
prompt_list = [
    {"role":"system", "content": sys_prompt},
    {"role": "user", "content": user_prompt.format(DIMENSION, DIMENSION, DIMENSION_RUBRIC[DIMENSION], DIMENSION, sub_questions)}
]

final_sub_questions = gpt4.ask_chatgpt(prompt_list)

final_sub_questions_list = ast.literal_eval(final_sub_questions)

In [18]:
checklist = ""

for sub_question in final_sub_questions_list:

    checklist+=f"- {sub_question}\n"

In [19]:
print(checklist)

- Does the summary start with an introduction that sets the context?
- Does the summary end with a conclusion that wraps up the main points?
- Are the ideas in the summary presented in a logical sequence?
- Does the summary avoid presenting information in a fragmented or disjointed manner?
- Does each sentence in the summary connect logically to the previous one?
- Are transitions effectively used to guide the reader from one idea to the next?
- Does the summary maintain a consistent flow of ideas and events throughout?
- Are the main points of the news article clearly and concisely represented in the summary?
- Does the summary avoid including unnecessary details?
- Is the language used in the summary clear and free from confusion?
- Does the summary maintain the same context as the original news article?
- Is the tone and style of the summary consistent with that of the original news article?
- Are pronouns and references used clearly and consistently in the summary?
- Does the summa

In [20]:
with open(f"./data/summeval/{DIMENSION}/{DIMENSION}_checklist.txt", "w") as file:
    file.write(checklist)